<a href="https://colab.research.google.com/github/fvangool/Deep-Learning-Specialization-Coursera/blob/main/GNN_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Run these first in Colab
!pip install torch-geometric torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install cuml-cu12  # This is the big one for KNN

Looking in links: https://data.pyg.org/whl/torch-2.4.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 117.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 109.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 69.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cuda-nvcc-cu12
    Found existing installation: nvidia-cuda-nvcc-cu12 12.5.82
    Uninstalling nvidia-cuda-nvcc-cu12-12.5.82:
      Successfully uninstalled nvidia-cuda-nvcc-cu12-12.5.82


In [7]:
#%%writefile gnn.py
"""
GNN — Irrigation Need (v23 style)
GraphSAGE + Categorical Embeddings | Adapted for your drive setup
"""

import gc
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, log_loss
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

# Try RAPIDS cuML, fallback to sklearn

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Use specific GPU if you have multiple

# Force cuML if available
try:
    from cuml.neighbors import NearestNeighbors
    print("🚀 Using RAPIDS cuML KNN - Very Fast!")
except ImportError:
    from sklearn.neighbors import NearestNeighbors
    print("⚠️ cuML not found → Falling back to sklearn KNN (slower)")

# ============================================================
# CONFIG — YOUR PARAMETERS
# ============================================================
OUT_DIR = "/content/drive/MyDrive/irrigation_need_v15/"
TRAIN_PATH = "/content/drive/MyDrive/irrigation_need_v15/train.csv"
TEST_PATH  = "/content/drive/MyDrive/irrigation_need_v15/test.csv"
VERSION_NB = "GNN_v24"
SEED = 42
N_FOLDS = 5
K = 8                    # KNN neighbors
EPOCHS = 80
PATIENCE = 15
BATCH_SIZE = 4096
INFER_BATCH = 8192
FANOUTS = [6, 4]
GRAPH_NUM_MULTIPLIER = 3.0
USE_AMP = True
RARE_MIN = 25
HIDDEN = 128
DROPOUT = 0.20
LR = 8e-4
WEIGHT_DECAY = 3e-4

TARGET = "Irrigation_Need"
LABEL_MAP = {"Low": 0, "Medium": 1, "High": 2}
LABEL_INV = {0: "Low", 1: "Medium", 2: "High"}
N_CLASSES = 3

NUMS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm"
]
CATS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region"
]

CAT_PROXY = [f"{c}__cat" for c in NUMS]
CAT_RARE = [f"{c}__is_rare" for c in NUMS]
ALL_CATS = CATS + CAT_PROXY + CAT_RARE
ALL_NUMS = NUMS[:]
GRAPH_CAT_COLS = CATS[:]
GRAPH_NUM_COLS = NUMS[:]

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {DEVICE}")

N_GPUS = torch.cuda.device_count()

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

from sklearn.metrics import balanced_accuracy_score

import numpy as np

def apply_thresholds(probs, thresholds):
    adjusted = probs - thresholds  # shift decision boundary
    return np.argmax(adjusted, axis=1)

from sklearn.metrics import balanced_accuracy_score

def fitness_thresholds(thresholds, probs, y):
    preds = apply_thresholds(probs, thresholds)
    return -balanced_accuracy_score(y, preds)  # maximize BA

import random

def optimize_thresholds(probs, y, pop_size=30, generations=40, mutation_scale=0.05):
    n_classes = probs.shape[1]

    # Initialize thresholds near 0
    population = [np.random.uniform(-0.2, 0.2, size=n_classes) for _ in range(pop_size)]

    for gen in range(generations):
        scores = np.array([fitness_thresholds(ind, probs, y) for ind in population])

        idx = np.argsort(scores)
        population = [population[i] for i in idx[:pop_size // 2]]

        children = []
        while len(children) < pop_size // 2:
            p1, p2 = random.sample(population, 2)
            child = (p1 + p2) / 2

            # Mutation
            child += np.random.normal(0, mutation_scale, size=n_classes)
            children.append(child)

        population.extend(children)

        print(f"Gen {gen:02d} | Best BA: {-scores[idx[0]]:.5f}")

    # Final best
    scores = np.array([fitness_thresholds(ind, probs, y) for ind in population])
    best = population[np.argmin(scores)]

    return best

# ===========================================================
# GENETIC BIAS HELPER FUNCTIONS
# ============================================================
import numpy as np

def apply_bias(probs, bias):
    logits = np.log(np.clip(probs, 1e-9, 1.0))
    logits = logits + bias
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)

from sklearn.metrics import log_loss

def fitness(bias, probs, y_true):
    adj = apply_bias(probs, bias)
    return log_loss(y_true, adj, labels=[0,1,2])

def genetic_optimize(probs, y, pop_size=30, generations=40, mutation_scale=0.1):
    n_classes = probs.shape[1]

    # Initialize population (bias vectors)
    population = [np.random.uniform(-0.5, 0.5, size=n_classes) for _ in range(pop_size)]

    for gen in range(generations):
        scores = np.array([fitness(ind, probs, y) for ind in population])

        # Select top 50%
        idx = np.argsort(scores)
        population = [population[i] for i in idx[:pop_size // 2]]

        # Reproduce
        children = []
        while len(children) < pop_size // 2:
            p1, p2 = random.sample(population, 2)
            child = (p1 + p2) / 2

            # Mutation
            child += np.random.normal(0, mutation_scale, size=n_classes)
            children.append(child)

        population.extend(children)

        print(f"Gen {gen:02d} | Best LogLoss: {scores[idx[0]]:.5f}")

    # Final best
    scores = np.array([fitness(ind, probs, y) for ind in population])
    best = population[np.argmin(scores)]

    return best

# REST OF YOUR ORIGINAL CODE (with small fixes)
# ============================================================
# ... [I kept all your original functions exactly the same] ...

def preprocess(train_df: pd.DataFrame, test_df: pd.DataFrame):
    tr = train_df.copy()
    te = test_df.copy()

    # --- NUMERIC FEATURES ---
    for c in NUMS:
        tr[c] = pd.to_numeric(tr[c], errors="coerce").astype(np.float32)
        te[c] = pd.to_numeric(te[c], errors="coerce").astype(np.float32)

        med = float(np.nanmedian(tr[c].values))
        tr[c] = tr[c].fillna(med)
        te[c] = te[c].fillna(med)

    # --- CATEGORICAL FEATURES ---
    for c in CATS:
        tr[c] = tr[c].astype(str).str.strip().fillna("missing")
        te[c] = te[c].astype(str).str.strip().fillna("missing")

    # --- LABELS (FINAL CORRECT VERSION) ---
    y = tr[TARGET].values.astype(np.int64)

    # Safety checks
    print("Unique labels:", np.unique(y))

    assert not np.isnan(y).any(), "NaNs in labels!"
    assert set(np.unique(y)) <= {0, 1, 2}, "Invalid labels detected!"

    return tr, te, y

# (All other functions _build_snapper, engineer_features, encode_categoricals,
#  scale_numerics, build_knn_graph, CatEmbed, IrrigationGNN, subgraph utils,
#  IrrigationGNNClassifier remain EXACTLY as you wrote them)
# ============================================================
# SECTION 3 — FEATURE ENGINEERING
# ============================================================

def _build_snapper(train_series: pd.Series):
    """
    Fit a rare-value snapper on train. Returns a transform function.
    Rare = appears fewer than RARE_MIN times.
    Rare values are snapped to the nearest frequent value.
    """
    s = pd.to_numeric(train_series, errors="coerce").astype(np.float32)
    vc = s.value_counts(dropna=False)
    frequent = np.sort(
        np.array([v for v in vc[vc >= RARE_MIN].index if pd.notna(v)], dtype=np.float32)
    )
    if frequent.size == 0:
        frequent = np.sort(s.dropna().unique().astype(np.float32))

    freq_set = set(frequent.tolist())

    def transform(series):
        x = pd.to_numeric(series, errors="coerce").astype(np.float32).values
        is_nan  = np.isnan(x)
        is_rare = np.ones(len(x), dtype=np.int32)

        for i, v in enumerate(x):
            if not np.isnan(v) and float(v) in freq_set:
                is_rare[i] = 0

        x_snapped = x.copy()
        snap_idx  = np.where((~is_nan) & (is_rare == 1))[0]
        if snap_idx.size > 0 and frequent.size > 0:
            v     = x[snap_idx]
            pos   = np.clip(np.searchsorted(frequent, v), 0, len(frequent) - 1)
            left  = np.clip(pos - 1, 0, len(frequent) - 1)
            right = pos
            nearest = np.where(
                np.abs(v - frequent[right]) <= np.abs(v - frequent[left]),
                frequent[right], frequent[left]
            )
            x_snapped[snap_idx] = nearest.astype(np.float32)

        return x_snapped.astype(np.float32), is_rare.astype(np.int32)

    return transform


def engineer_features(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    For each numeric column: add __cat (snapped value as string) and __is_rare flag.
    Snappers are fit on train only.
    """
    print("\n[Feature Engineering] Building rare-snap features...")
    tr = train_df.copy()
    te = test_df.copy()

    for col in NUMS:
        snapper = _build_snapper(tr[col])
        tr_snap, tr_rare = snapper(tr[col])
        te_snap, te_rare = snapper(te[col])

        tr[f"{col}__cat"]     = pd.Series(tr_snap).astype(str).values
        te[f"{col}__cat"]     = pd.Series(te_snap).astype(str).values
        tr[f"{col}__is_rare"] = pd.Series(tr_rare).astype(str).values
        te[f"{col}__is_rare"] = pd.Series(te_rare).astype(str).values

    for df in (tr, te):
        for c in ALL_CATS:
            df[c] = df[c].astype(str).fillna("missing")

    print(f"[Feature Engineering] Node categorical features : {len(ALL_CATS)}")
    print(f"[Feature Engineering] Node numeric features     : {len(ALL_NUMS)}")
    return tr, te


def encode_categoricals(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    Integer-encode all categorical node features using a shared vocab
    built from train + test (no unseen categories at inference).
    Returns integer matrices and per-feature cardinalities.
    """
    print("\n[Encode] Encoding categorical node features...")
    tr_codes, te_codes, cardinalities = [], [], []

    for c in ALL_CATS:
        all_vals = pd.concat(
            [train_df[c].astype(str), test_df[c].astype(str)], ignore_index=True
        )
        mapping = {v: i for i, v in enumerate(all_vals.unique())}
        tr_codes.append(train_df[c].astype(str).map(mapping).fillna(0).astype(np.int64).values)
        te_codes.append(test_df[c].astype(str).map(mapping).fillna(0).astype(np.int64).values)
        cardinalities.append(len(mapping))

    Xc_tr = np.stack(tr_codes, axis=1)
    Xc_te = np.stack(te_codes, axis=1)
    print(f"[Encode] Categorical matrix — train: {Xc_tr.shape} | test: {Xc_te.shape}")
    return Xc_tr, Xc_te, cardinalities


def scale_numerics(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """StandardScale numeric node features, fit on train."""
    print("\n[Scale] Scaling numeric node features...")
    scaler = StandardScaler()
    Xn_tr = scaler.fit_transform(
        train_df[ALL_NUMS].values.astype(np.float32)
    ).astype(np.float32)
    Xn_te = scaler.transform(
        test_df[ALL_NUMS].values.astype(np.float32)
    ).astype(np.float32)
    print(f"[Scale] Numeric matrix — train: {Xn_tr.shape} | test: {Xn_te.shape}")
    return Xn_tr, Xn_te


def build_knn_graph(train_df: pd.DataFrame, test_df: pd.DataFrame, k: int = K):
    """
    Build KNN graph on combined train+test using cuML.
    Graph features: OHE of base categoricals + standardized numerics * multiplier.
    Returns neighbors array of shape [n_total, k].
    """
    print(f"\n[Graph] Building KNN graph (k={k}) on {len(train_df) + len(test_df):,} nodes with cuML...")

    graph_cat = pd.concat(
        [train_df[GRAPH_CAT_COLS].astype(str), test_df[GRAPH_CAT_COLS].astype(str)],
        ignore_index=True
    )
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float32)
    X_cat_ohe = ohe.fit_transform(graph_cat).astype(np.float32)

    graph_num_tr = train_df[GRAPH_NUM_COLS].copy()
    graph_num_te = test_df[GRAPH_NUM_COLS].copy()
    for c in GRAPH_NUM_COLS:
        graph_num_tr[c] = pd.to_numeric(graph_num_tr[c], errors="coerce").fillna(0).astype(np.float32)
        graph_num_te[c] = pd.to_numeric(graph_num_te[c], errors="coerce").fillna(0).astype(np.float32)

    num_scaler = StandardScaler()
    X_num_tr = num_scaler.fit_transform(graph_num_tr.values.astype(np.float32))
    X_num_te = num_scaler.transform(graph_num_te.values.astype(np.float32))
    X_num    = np.vstack([X_num_tr, X_num_te]).astype(np.float32) * GRAPH_NUM_MULTIPLIER

    X_graph = np.concatenate([X_cat_ohe, X_num], axis=1).astype(np.float32)
    print(f"[Graph] Graph feature matrix shape: {X_graph.shape}")

    knn = NearestNeighbors(n_neighbors=k)
    knn.fit(X_graph)
    _, idx = knn.kneighbors(X_graph)

    # Handle cuML vs sklearn
    if hasattr(idx, "get"):
        idx = idx.get()

    neighbors = idx.astype(np.int32)

    print(f"[Graph] Neighbors matrix shape: {neighbors.shape}")
    del X_graph, X_cat_ohe, X_num, idx, knn
    gc.collect()

    return neighbors
# ============================================================
# SECTION 4 — MODEL DEFINITION
# ============================================================

def _emb_dim(cardinality: int) -> int:
    return int(np.clip(round(1.8 * (cardinality ** 0.25)), 4, 24))


class CatEmbed(nn.Module):
    def __init__(self, cardinalities):
        super().__init__()
        self.embs = nn.ModuleList()
        self.out_dim = 0
        for card in cardinalities:
            card = max(2, int(card))
            d = _emb_dim(card)
            self.embs.append(nn.Embedding(card, d))
            self.out_dim += d
        for e in self.embs:
            nn.init.normal_(e.weight, 0.0, 0.02)

    def forward(self, x_cat):
        return torch.cat([emb(x_cat[:, j]) for j, emb in enumerate(self.embs)], dim=1)


class IrrigationGNN(nn.Module):
    """
    2-layer GraphSAGE with categorical embeddings.
    Residual connections + LayerNorm between SAGE layers.
    Output: 3-class logits.
    """
    def __init__(self, num_in: int, cardinalities: list, hidden: int = 128, dropout: float = 0.2):
        super().__init__()
        self.cat    = CatEmbed(cardinalities)
        in_dim      = num_in + self.cat.out_dim

        self.lin_in = nn.Linear(in_dim, hidden)
        self.conv1  = SAGEConv(hidden, hidden)
        self.conv2  = SAGEConv(hidden, hidden)
        self.norm1  = nn.LayerNorm(hidden)
        self.norm2  = nn.LayerNorm(hidden)
        self.drop   = dropout

        self.head = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, N_CLASSES),
        )

    def forward(self, data):
        x = torch.cat([data.x_num, self.cat(data.x_cat)], dim=1)
        x = F.dropout(F.relu(self.lin_in(x)), p=self.drop, training=self.training)

        x1 = F.relu(self.norm1(self.conv1(x,  data.edge_index)))
        x1 = F.dropout(x1, p=self.drop, training=self.training)
        x  = x + 0.5 * x1

        x2 = F.relu(self.norm2(self.conv2(x, data.edge_index)))
        x2 = F.dropout(x2, p=self.drop, training=self.training)
        x  = x + 0.5 * x2

        return self.head(x)   # [N, 3]


# ============================================================
# SECTION 5 — SUBGRAPH SAMPLING UTILITIES
# ============================================================

_global_pos = None   # reused buffer, initialised per-fold inside fit()


def _build_subgraph(seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu, fanouts, device, offset=0):
    global _global_pos
    seed_nodes = np.asarray(seed_nodes, dtype=np.int32)
    frontier   = seed_nodes
    collected  = [seed_nodes]

    for hop, fanout in enumerate(fanouts):
        nbr   = neighbors[frontier]
        start = (offset + hop) % nbr.shape[1]
        cols  = (np.arange(fanout) + start) % nbr.shape[1]
        frontier = np.unique(nbr[:, cols].reshape(-1))
        collected.append(frontier)

    nodes = np.unique(np.concatenate(collected))
    m     = len(nodes)
    _global_pos[nodes] = np.arange(m, dtype=np.int32)

    sub_nbr      = neighbors[nodes]
    dst_local    = _global_pos[sub_nbr]
    mask         = dst_local >= 0
    src_l        = np.repeat(np.arange(m, dtype=np.int64), sub_nbr.shape[1])[mask.reshape(-1)]
    dst_l        = dst_local[mask].astype(np.int64)
    edge_index   = torch.tensor(np.vstack([src_l, dst_l]), dtype=torch.long, device=device)

    batch = Data(
        x_num      = x_num_cpu[nodes].to(device),
        x_cat      = x_cat_cpu[nodes].to(device, non_blocking=True),
        y          = y_cpu[nodes].to(device, non_blocking=True),
        edge_index = edge_index,
    )
    batch.seed_local = torch.tensor(_global_pos[seed_nodes], dtype=torch.long, device=device)
    _global_pos[nodes] = -1
    return batch


def _seed_batches(seed_nodes, batch_size, shuffle):
    arr = np.asarray(seed_nodes, dtype=np.int32).copy()
    if shuffle:
        np.random.shuffle(arr)
    for i in range(0, len(arr), batch_size):
        yield arr[i:i + batch_size]


@torch.no_grad()
def _predict_nodes(model, seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu,
                   fanouts, batch_size, device, offset=0):
    model.eval()
    out = np.zeros((len(seed_nodes), N_CLASSES), dtype=np.float32)
    pos = 0
    for batch_seeds in _seed_batches(seed_nodes, batch_size, shuffle=False):
        batch  = _build_subgraph(batch_seeds, neighbors, x_num_cpu, x_cat_cpu,
                                  y_cpu, fanouts, device, offset)
        with torch.autocast(device_type="cuda", dtype=torch.float16,
                            enabled=(USE_AMP and device.type == "cuda")):
            logits = model(batch)
        probs = F.softmax(logits[batch.seed_local], dim=1).float().cpu().numpy()
        out[pos:pos + len(batch_seeds)] = probs
        pos += len(batch_seeds)
        del batch, logits, probs
    return out   # [n_seeds, 3]


# ============================================================
# SECTION 6 — SKLEARN-STYLE WRAPPER
# ============================================================

class IrrigationGNNClassifier(BaseEstimator, ClassifierMixin):
    """
    Sklearn-style wrapper for the 3-class GNN.

    Parameters
    ----------
    hidden      : hidden dimension for SAGE layers
    dropout     : dropout rate
    lr          : AdamW learning rate
    weight_decay: AdamW weight decay
    epochs      : max training epochs
    patience    : early stopping patience (on val log-loss, lower = better)
    batch_size  : mini-batch seed size during training
    infer_batch : mini-batch seed size during inference
    fanouts     : neighbourhood fanouts per SAGE hop
    device      : torch device
    use_amp     : mixed precision
    """
    def __init__(
        self,
        hidden       = 128,
        dropout      = 0.20,
        lr           = 1e-3,
        weight_decay = 3e-4,
        epochs       = EPOCHS,
        patience     = PATIENCE,
        batch_size   = BATCH_SIZE,
        infer_batch  = INFER_BATCH,
        fanouts      = FANOUTS,
        device       = DEVICE,
        use_amp      = USE_AMP,
        n_gpus       = None,
    ):
        self.hidden       = hidden
        self.dropout      = dropout
        self.lr           = lr
        self.weight_decay = weight_decay
        self.epochs       = epochs
        self.patience     = patience
        self.batch_size   = batch_size
        self.infer_batch  = infer_batch
        self.fanouts      = fanouts
        self.device       = device
        self.use_amp      = use_amp

        self.model_        = None
        self.cardinalities_= None
        self.n_gpus = n_gpus if n_gpus is not None else torch.cuda.device_count()

    def fit(
        self,
        train_idx,
        val_idx,
        y_all,
        neighbors,
        x_num_cpu,
        x_cat_cpu,
        y_cpu,
        cardinalities,
        class_weights,

        #if self.n_gpus > 1::
         #   self.device = torch.device("cuda:0")
    ):
        """
        Train on train_idx nodes, validate on val_idx nodes.
        All node features live in the shared x_num_cpu / x_cat_cpu tensors (train+test).

        Parameters
        ----------
        train_idx     : np.ndarray of global node indices for this fold's train split
        val_idx       : np.ndarray of global node indices for this fold's val split
        y_all         : np.ndarray, integer labels for all training nodes (-1 for test)
        neighbors     : KNN neighbors array [n_total, K]
        x_num_cpu     : pinned CPU tensor [n_total, n_num_features]
        x_cat_cpu     : pinned CPU tensor [n_total, n_cat_features]
        y_cpu         : pinned CPU tensor [n_total] (float, -1 for test)
        cardinalities : list of ints, cardinality per categorical feature
        class_weights : torch.Tensor of shape [N_CLASSES], balanced class weights
        """
        global _global_pos
        n_all = x_num_cpu.shape[0]
        _global_pos = np.full(n_all, -1, dtype=np.int32)

        self.cardinalities_ = cardinalities

        # Build model; wrap with DataParallel if multiple GPUs available
        core_model = IrrigationGNN(
            num_in       = x_num_cpu.shape[1],
            cardinalities= cardinalities,
            hidden       = self.hidden,
            dropout      = self.dropout,
        )
        if self.n_gpus > 1:
            core_model = nn.DataParallel(core_model)
        self.model_ = core_model.to(self.device)
        N_GPUS = torch.cuda.device_count()

        loss_fn = nn.CrossEntropyLoss(
            weight=class_weights.to(self.device)
        )
        opt     = torch.optim.AdamW(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scaler  = scaler = torch.cuda.amp.GradScaler(enabled=(self.use_amp and self.device.type == "cuda"))

        best_val_loss = float("inf")
        best_state    = None
        bad_epochs    = 0

        print(f"    Model parameters : {sum(p.numel() for p in self.model_.parameters()):,}")
        print(f"    Training nodes   : {len(train_idx):,}  |  Validation nodes: {len(val_idx):,}")
        print(f"    Max epochs: {self.epochs}  |  Patience: {self.patience}\n")

        for epoch in range(1, self.epochs + 1):
            # ---- train ----
            self.model_.train()
            epoch_losses = []
            offset = epoch % K

            for batch_seeds in _seed_batches(train_idx, self.batch_size, shuffle=True):
                batch = _build_subgraph(
                    batch_seeds, neighbors, x_num_cpu, x_cat_cpu,
                    y_cpu, self.fanouts, self.device, offset
                )
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type="cuda", dtype=torch.float16,
                                    enabled=(self.use_amp and self.device.type == "cuda")):
                    logits = self.model_(batch)
                    loss   = loss_fn(logits[batch.seed_local], batch.y[batch.seed_local].long())
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(self.model_.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                epoch_losses.append(loss.item())
                del batch, logits, loss

            # ---- validate ----
            val_probs = _predict_nodes(
                self.model_, val_idx, neighbors, x_num_cpu, x_cat_cpu,
                y_cpu, self.fanouts, self.infer_batch, self.device, offset
            )
            y_val_true = y_all[val_idx]
            val_loss   = log_loss(y_val_true, val_probs, labels=[0, 1, 2])
            val_bal_acc = balanced_accuracy_score(y_val_true, np.argmax(val_probs, axis=1))

            print(
                f"    Epoch {epoch:04d} | "
                f"Train Loss: {np.mean(epoch_losses):.5f} | "
                f"Val Log-Loss: {val_loss:.5f} | "
                f"Val Bal-Acc: {val_bal_acc:.5f}"
            )

            # ---- early stopping ----
            if val_loss < best_val_loss - 1e-6:
                best_val_loss = val_loss
                best_state    = {k: v.detach().cpu().clone() for k, v in self.model_.state_dict().items()}
                bad_epochs    = 0
            else:
                bad_epochs += 1
                if bad_epochs >= self.patience:
                    print(f"\n    [Early Stop] No improvement for {self.patience} epochs. Stopping at epoch {epoch}.")
                    break

        print(f"\n    [Best] Val Log-Loss: {best_val_loss:.5f}")
        if best_state is not None:
          self.model_.load_state_dict(best_state)
        return self

    def predict_proba(self, seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu):
        """Returns softmax probabilities of shape [n_seeds, N_CLASSES]."""
        return _predict_nodes(
            self.model_, seed_nodes, neighbors, x_num_cpu, x_cat_cpu,
            y_cpu, self.fanouts, self.infer_batch, self.device
        )

    def predict(self, seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu):
        """Returns integer class predictions."""
        probs = self.predict_proba(seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu)
        return np.argmax(probs, axis=1)


# ============================================================
# SECTION 7 — LOAD DATA & BUILD GRAPH
# ============================================================

print("\n" + "="*60)
print("Loading data...")
print("="*60)

#train_raw = pd.read_csv(TRAIN_PATH)
#test_raw  = pd.read_csv(TEST_PATH)
TRAIN_PARQUET = "/content/drive/MyDrive/irrigation_need_v15/train_engineered_v21.parquet"
TEST_PARQUET  = "/content/drive/MyDrive/irrigation_need_v15/test_engineered_v21.parquet"

train_raw = pd.read_parquet(TRAIN_PARQUET)
test_raw  = pd.read_parquet(TEST_PARQUET)

print(f"Raw train: {train_raw.shape} | Raw test: {test_raw.shape}")

train_pre, test_pre, y_train = preprocess(train_raw, test_raw)
train_fe,  test_fe           = engineer_features(train_pre, test_pre)

Xc_train, Xc_test, cat_cardinalities = encode_categoricals(train_fe, test_fe)
Xn_train, Xn_test                    = scale_numerics(train_fe, test_fe)

neighbors = build_knn_graph(train_fe, test_fe, k=K)

n_train = len(train_fe)
n_test  = len(test_fe)
n_all   = n_train + n_test

# Combine train + test node features into shared tensors
Xn_all = np.vstack([Xn_train, Xn_test])
Xc_all = np.vstack([Xc_train, Xc_test])

y_all_np = np.concatenate([y_train, np.full(n_test, -1, dtype=np.int64)])

x_num_cpu = torch.tensor(Xn_all, dtype=torch.float32).pin_memory()
x_cat_cpu = torch.tensor(Xc_all, dtype=torch.long).pin_memory()
y_cpu     = y_cpu = torch.tensor(y_all_np, dtype=torch.long).pin_memory()

print(f"\n[Tensors] x_num: {tuple(x_num_cpu.shape)} | x_cat: {tuple(x_cat_cpu.shape)}")

# Class weights (balanced) from training labels
classes       = np.arange(N_CLASSES)
cw_values     = compute_class_weight("balanced", classes=classes, y=y_train)
class_weights = torch.tensor(cw_values, dtype=torch.float32)
print(f"\n[Class Weights] { {LABEL_INV[i]: round(float(w), 4) for i, w in enumerate(cw_values)} }")


# ============================================================
# SECTION 8 — CROSS-VALIDATION TRAINING
# ============================================================

print("\n" + "="*60)
print("Starting 5-Fold Cross-Validation")
print("="*60)

skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_probs  = np.zeros((n_train, N_CLASSES), dtype=np.float32)
pred_probs = np.zeros((n_test,  N_CLASSES), dtype=np.float32)
test_nodes = np.arange(n_train, n_all, dtype=np.int32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(n_train), y_train), 1):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold} / {N_FOLDS}")
    print(f"{'='*60}\n")

    t_fold = time.time()

    clf = IrrigationGNNClassifier()
    clf.fit(
        train_idx    = tr_idx,
        val_idx      = va_idx,
        y_all        = y_all_np,
        neighbors    = neighbors,
        x_num_cpu    = x_num_cpu,
        x_cat_cpu    = x_cat_cpu,
        y_cpu        = y_cpu,
        cardinalities= cat_cardinalities,
        class_weights= class_weights,
    )

    # OOF predictions
    oof_probs[va_idx] = clf.predict_proba(
        va_idx, neighbors, x_num_cpu, x_cat_cpu, y_cpu
    )

    # Test predictions (averaged across folds)
    pred_probs += clf.predict_proba(
        test_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu
    ) / N_FOLDS

    fold_bal_acc = balanced_accuracy_score(y_train[va_idx], np.argmax(oof_probs[va_idx], axis=1))
    fold_logloss = log_loss(y_train[va_idx], oof_probs[va_idx], labels=[0, 1, 2])

    print(f"\n  [Fold {fold} Result] Bal-Acc: {fold_bal_acc:.5f} | Log-Loss: {fold_logloss:.5f} | Time: {time.time()-t_fold:.1f}s")
    print(f"\n{'='*60}\n\n")

    del clf
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# SECTION 9 — OOF EVALUATION
# ============================================================

oof_preds    = np.argmax(oof_probs, axis=1)
cv_bal_acc   = balanced_accuracy_score(y_train, oof_preds)
cv_logloss   = log_loss(y_train, oof_probs, labels=[0, 1, 2])

#treshold tuning
from sklearn.metrics import balanced_accuracy_score

best_thresholds = optimize_thresholds(oof_probs, y_train)
print("Best thresholds:", best_thresholds)

# Raw predictions (no tuning)
oof_preds_raw = np.argmax(oof_probs, axis=1)
ba_raw = balanced_accuracy_score(y_train, oof_preds_raw)

# Threshold-tuned predictions
oof_preds_thresh = apply_thresholds(oof_probs, best_thresholds)
ba_thresh = balanced_accuracy_score(y_train, oof_preds_thresh)

print(f"Raw BA       : {ba_raw:.5f}")
print(f"Threshold BA : {ba_thresh:.5f}")
print(f"Gain         : {ba_thresh - ba_raw:.5f}")

test_preds_thresh = apply_thresholds(pred_probs, best_thresholds)
np.save(f"{OUT_DIR}/thresholds_{VERSION_NB}.npy", best_thresholds)
np.save(f"{OUT_DIR}/test_preds_thresh_{VERSION_NB}.npy", test_preds_thresh)

#genetic tuning
best_bias = genetic_optimize(oof_probs, y_train)
print("Best bias:", best_bias)
final_test_probs = apply_bias(pred_probs, best_bias)
final_preds = np.argmax(final_test_probs, axis=1)
oof_probs_tuned = apply_bias(oof_probs, best_bias)


np.save(f"{OUT_DIR}/test_preds_tuned_{VERSION_NB}.npy", final_test_probs)
#np.save(f"{OUT_DIR}/oof_{VERSION_NB}.npy", oof_probs)
np.save(f"{OUT_DIR}/oof_tuned_{VERSION_NB}.npy", oof_probs_tuned)

oof_preds_tuned = np.argmax(oof_probs_tuned, axis=1)

from sklearn.metrics import balanced_accuracy_score



print("\n" + "="*60)
print("  OOF EVALUATION")
print("="*60)
print(f"  CV Balanced Accuracy : {cv_bal_acc:.5f}")
print(f"  CV Log-Loss          : {cv_logloss:.5f}")

ba_tuned = balanced_accuracy_score(y_train, oof_preds_tuned)
print(f"Tuned Balanced Accuracy: {ba_tuned:.5f}")




# ============================================================
# SECTION 10 — SUBMISSION
# ============================================================

print("\n[Submission] Generating predictions...")

test_preds      = np.argmax(pred_probs, axis=1)
test_labels     = [LABEL_INV[p] for p in test_preds]

submission = pd.DataFrame({
    "id"           : test_raw["id"],
    TARGET         : test_labels,
})
submission.to_csv(f"submission_gnn_{VERSION_NB}.csv", index=False)


🚀 Using RAPIDS cuML KNN - Very Fast!
DEVICE: cuda

Loading data...
Raw train: (640000, 111) | Raw test: (270000, 110)
Unique labels: [0 1 2]

[Feature Engineering] Building rare-snap features...
[Feature Engineering] Node categorical features : 30
[Feature Engineering] Node numeric features     : 11

[Encode] Encoding categorical node features...
[Encode] Categorical matrix — train: (640000, 30) | test: (270000, 30)

[Scale] Scaling numeric node features...
[Scale] Numeric matrix — train: (640000, 11) | test: (270000, 11)

[Graph] Building KNN graph (k=8) on 910,000 nodes with cuML...
[Graph] Graph feature matrix shape: (910000, 43)
[Graph] Neighbors matrix shape: (910000, 8)

[Tensors] x_num: (910000, 11) | x_cat: (910000, 30)

[Class Weights] {'Low': 0.5677, 'Medium': 0.8784, 'High': 9.9945}

Starting 5-Fold Cross-Validation

  FOLD 1 / 5

    Model parameters : 467,730
    Training nodes   : 512,000  |  Validation nodes: 128,000
    Max epochs: 80  |  Patience: 15

    Epoch 0001 | 

In [6]:
print("Unmapped values:", y_raw[y.isna()].unique())

NameError: name 'y_raw' is not defined

In [10]:
np.save(f"{OUT_DIR}/oof_{VERSION_NB}.npy", oof_probs)
np.save(f"{OUT_DIR}/test_preds_{VERSION_NB}.npy", pred_probs)
np.save(f"{OUT_DIR}/y_train_{VERSION_NB}.npy", y_train)

In [9]:
print(train_raw[TARGET].value_counts())

Irrigation_Need
0    375781
1    242874
2     21345
Name: count, dtype: int64


In [8]:
print("Unique labels in y_train:", np.unique(y_train))

Unique labels in y_train: [-9223372036854775808]
